In [0]:
-- NVIRONMENTAL PROFILE BY SPECIES AND LOCALITY         
-- What conditions was each species experiencing? 


-- What to look for:
-- Do different species occupy different temperature ranges?
-- Which locality has the highest average chlorophyll (most food)?
-- Which species has the highest habitat stress on average?
 
SELECT
species_common,
locality,
count(*) AS total_observations,
round(avg(sst), 2) AS avg_sst,
round(min(sst), 2) AS min_sst,
round(max(sst), 2) AS max_sst,
round(avg(cc), 3) AS avg_chlorophyll,
round(min(cc), 3) AS min_chlorophyll,
round(max(cc), 3) AS max_chlorophyll,
round(avg(habitat_stress_index), 2) AS avg_stress_index
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
GROUP BY species_common, locality
ORDER BY species_common, locality;

In [0]:
-- ENSO PHASE EFFECT ON EACH SPECIES  
-- Do animals experience different conditions during El Nino?  

-- What to look for:
-- Is average SST higher during El Nino than La Nina?
-- Is chlorophyll lower during El Nino? (suppressed upwelling = less food)
-- Does habitat stress increase during El Nino events?
 
SELECT
species_common,
enso_phase,
count(*) AS observations,
round(avg(sst), 2) AS avg_sst,
round(avg(cc), 3) AS avg_chlorophyll,
round(avg(habitat_stress_index), 2) AS avg_stress_index
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
AND enso_phase IS NOT NULL
GROUP BY species_common, enso_phase
ORDER BY species_common, enso_phase;

In [0]:
-- SEASONAL PATTERNS BY SPECIES 
-- Do conditions vary by season for each species?



SELECT
species_common,
locality,
season,
count(*) AS observations,
round(avg(sst), 2) AS avg_sst,
round(avg(cc), 3) AS avg_chlorophyll
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
AND season IS NOT NULL
GROUP BY species_common, locality, season
ORDER BY species_common, locality,
CASE season
WHEN 'Winter' THEN 1
WHEN 'Spring' THEN 2
WHEN 'Summer' THEN 3
WHEN 'Autumn' THEN 4
END;

In [0]:
-- HABITAT STRESS INDEX BREAKDOWN
-- How stressed were each species habitats?    

-- What to look for:
-- Which species had the most observations at stress level 2 or 3?
-- Is any species spending significant time in highly stressed habitat?
 
SELECT
species_common,
habitat_stress_index,
count(*) AS observations,
round(
count(*) * 100.0 / sum(count(*)) OVER (PARTITION BY species_common),
1) AS pct_of_species_total,
round(avg(sst), 2) AS avg_sst,
round(avg(cc), 3) AS avg_chlorophyll
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE habitat_stress_index IS NOT NULL
AND sst IS NOT NULL
GROUP BY species_common, habitat_stress_index
ORDER BY species_common, habitat_stress_index;

In [0]:
-- ARE FEEDING GROUNDS ALSO POLLUTION HOTSPOTS? 
-- The key pollution question for filter feeders      

-- What to look for:
-- Do rows with highest avg_chlorophyll also show high avg_microplastics?
-- If yes — filter feeders like manta rays and whale sharks are being
-- exposed to plastic pollution exactly where they go to feed
 
SELECT
h3_res4,
round(avg(sea_surface_temperature), 2) AS avg_sst,
round(avg(chlorophyll_concentration), 3) AS avg_chlorophyll,
round(avg(microplastic_concentration), 4) AS avg_microplastics,
pollution_severity,
count(*) AS months_with_data
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE microplastic_concentration IS NOT NULL
AND chlorophyll_concentration IS NOT NULL
GROUP BY h3_res4, pollution_severity
ORDER BY avg_chlorophyll DESC
LIMIT 50;

In [0]:
-- YEAR BY YEAR SST TREND 
-- Is the ocean warming over the study period?   

-- What to look for:
-- Is average SST in 2023-2025 higher than in 2009-2010?
-- The 2015 column should be notably warm (strong El Nino event)
 
SELECT
year,
count(*) AS grid_cells,
round(avg(sst), 2) AS avg_sst,
round(min(sst), 2) AS min_sst,
round(max(sst), 2) AS max_sst,
round(avg(cc), 3) AS avg_chlorophyll,
mode() WITHIN GROUP (ORDER BY enso_phase) AS dominant_enso_phase
FROM mcphersonsharyn_gold.gold_environment_monthly
GROUP BY year
ORDER BY year;

In [0]:
-- HIGHEST STRESS H3 CELLS
-- Which locations are most degraded? Use in Tableau hex map  

-- Cells with avg_stress near 3 are priority areas for conservation
-- Use in Tableau: colour hex cells by avg_stress — dark = most degraded
 
SELECT
h3_res4,
h3_res5,
count(*) AS animal_observations,
round(avg(habitat_stress_index), 2) AS avg_stress,
round(avg(sst), 2) AS avg_sst,
round(avg(cc), 3) AS avg_chlorophyll,
round(avg(microplastic_concentration), 4) AS avg_microplastics,
collect_set(species_common) AS species_present
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE habitat_stress_index IS NOT NULL
GROUP BY h3_res4, h3_res5
HAVING count(*) >= 5
ORDER BY avg_stress DESC
LIMIT 50;



In [0]:
-- CROSS OCEAN BASIN COMPARISON
-- Different species, different oceans — how do conditions compare? 

-- What to look for:
-- Which ocean basin has the warmest SST on average?
-- Which has the most chlorophyll (most productive ocean)?
-- Note: microplastics is Atlantic only so NULLs are expected for
-- Indonesian and Indian Ocean cells
 
SELECT
species_common,
locality,
count(DISTINCT year) AS years_of_data,
round(avg(sst), 2) AS avg_sst,
round(stddev(sst), 3) AS sst_variability,
round(avg(cc), 3) AS avg_chlorophyll,
count(*) filter(WHERE microplastic_concentration IS NOT NULL) AS obs_with_pollution_data,
round(avg(microplastic_concentration), 4) AS avg_microplastics
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
GROUP BY species_common, locality
ORDER BY avg_sst DESC;

In [0]:

-- ENSO EFFECT ON CHLOROPHYLL
-- Does El Nino reduce food availability for filter feeders? 

-- El Nino suppresses upwelling in many tropical regions which means
-- fewer nutrients reach the surface = less phytoplankton = less food.
-- This query tests whether chlorophyll is lower during El Nino years.
 
SELECT
enso_phase,
species_common,
count(*) AS observations,
round(avg(sst), 2) AS avg_sst,
round(avg(cc), 4) AS avg_chlorophyll,
round(min(cc), 4) AS min_chlorophyll,
round(max(cc), 4) AS max_chlorophyll
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE enso_phase IS NOT NULL
AND cc IS NOT NULL
AND sst IS NOT NULL
GROUP BY enso_phase, species_common
ORDER BY species_common, enso_phase;

In [0]:
-- Complete environmental context for all species 


SELECT
species_common,
species_scientific,
locality,
season,
enso_phase,
count(*) AS observations,
count(DISTINCT animal_id) AS unique_animals,
count(DISTINCT year) AS years_covered,
round(avg(sst), 2) AS avg_sst,
round(min(sst), 2) AS min_sst,
round(max(sst), 2) AS max_sst,
round(avg(cc), 3) AS avg_chlorophyll,
round(avg(microplastic_concentration), 4) AS avg_microplastics,
round(avg(habitat_stress_index), 2) AS avg_habitat_stress,
round(count(*) filter(WHERE habitat_stress_index = 0) * 100.0 / count(*), 1) AS pct_stress_0,
round(count(*) filter(WHERE habitat_stress_index = 1) * 100.0 / count(*), 1) AS pct_stress_1,
round(count(*) filter(WHERE habitat_stress_index = 2) * 100.0 / count(*), 1) AS pct_stress_2,
round(count(*) filter(WHERE habitat_stress_index = 3) * 100.0 / count(*), 1) AS pct_stress_3
FROM mcphersonsharyn_gold.migration_environmental_joined
WHERE sst IS NOT NULL
AND season IS NOT NULL
AND enso_phase IS NOT NULL
GROUP BY species_common, species_scientific, locality, season, enso_phase
ORDER BY species_common, locality, season, enso_phase;

In [0]:
-- Year on year change
-- Shows increase or decrease in unique animals compared to previous year
-- Negative change_in_animals = fewer animals than previous year
WITH yearly AS (
SELECT
species_common,
locality,
year,
count(DISTINCT animal_id) AS unique_animals
FROM mcphersonsharyn_silver.migration_silver
GROUP BY species_common, locality, year
)
SELECT
species_common,
locality,
year,
unique_animals,
LAG(unique_animals) OVER (
PARTITION BY species_common, locality
ORDER BY year
) AS previous_year_animals,
unique_animals - LAG(unique_animals) OVER (
PARTITION BY species_common, locality
ORDER BY year
) AS change_in_animals,
CASE
WHEN unique_animals < LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year)
THEN 'Decline'
WHEN unique_animals > LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year)
THEN 'Increase'
WHEN unique_animals = LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year)
THEN 'No change'
ELSE 'First year'
END AS trend
FROM yearly
ORDER BY species_common, locality, year;

In [0]:

-- Year by year species presence with environmental context
WITH yearly AS (
SELECT
m.species_common,
m.locality,
m.year,
count(DISTINCT m.animal_id) AS unique_animals,
round(avg(e.sea_surface_temperature), 2) AS avg_sst,
round(avg(e.chlorophyll_concentration), 4) AS avg_cc,
mode() WITHIN GROUP (ORDER BY e.enso_phase) AS dominant_enso_phase
FROM mcphersonsharyn_silver.migration_silver m
LEFT JOIN mcphersonsharyn_silver.environmental_conditions_silver e
ON m.h3_res4 = e.h3_res4
AND m.year = e.year
AND m.month = e.month
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY m.species_common, m.locality, m.year
)
SELECT
species_common,
locality,
year,
unique_animals,
avg_sst,
avg_cc,
dominant_enso_phase,
LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year
) AS previous_year_animals,
unique_animals - LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year
) AS change_in_animals,
CASE
WHEN unique_animals < LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year)
THEN 'Decline'
WHEN unique_animals > LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year)
THEN 'Increase'
WHEN unique_animals = LAG(unique_animals) OVER (
PARTITION BY species_common, locality ORDER BY year)
THEN 'No change'
ELSE 'First year'
END AS trend
FROM yearly
ORDER BY species_common, locality, year;

In [0]:

-- ATLAS — Long term pattern analysis
-- Testing whether warming trend holds across all available years


-- Query 1: Full SST trend for Raja Ampat across all years
-- regardless of whether animals were present
-- This shows the environmental baseline shifting over time

SELECT
year,
month,
round(avg(sea_surface_temperature), 2) AS avg_sst,
round(avg(chlorophyll_concentration), 4) AS avg_cc,
enso_phase,
count(*) AS anchor_points
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE h3_res4 IN (
-- Get all H3 cells where manta rays were observed in Raja Ampat
SELECT DISTINCT h3_res4
FROM mcphersonsharyn_silver.manta_ray_silver
WHERE locality = 'Raja Ampat'
)
GROUP BY year, month, enso_phase
ORDER BY year, month;


-- ─────────────────────────────────────────────────────────────
-- Query 2: Annual SST average in Raja Ampat H3 cells
-- compared to ENSO phase — does warming persist regardless
-- of El Nino or La Nina?
-- ─────────────────────────────────────────────────────────────

SELECT
year,
enso_phase,
round(avg(sea_surface_temperature), 2) AS avg_sst,
round(avg(chlorophyll_concentration), 4) AS avg_cc,
count(*) AS anchor_points
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE h3_res4 IN (
SELECT DISTINCT h3_res4
FROM mcphersonsharyn_silver.manta_ray_silver
WHERE locality = 'Raja Ampat'
)
GROUP BY year, enso_phase
ORDER BY year;


-- ─────────────────────────────────────────────────────────────
-- Query 3: The core pattern test
-- Compare average SST during El Nino vs La Nina vs Neutral
-- for each year — if later La Nina years are warmer than
-- earlier El Nino years that confirms the warming signal
-- is stronger than the ENSO cycle
-- ─────────────────────────────────────────────────────────────

WITH enso_sst AS (
SELECT
year,
enso_phase,
round(avg(sea_surface_temperature), 2) AS avg_sst,
round(avg(chlorophyll_concentration), 4) AS avg_cc
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE h3_res4 IN (
SELECT DISTINCT h3_res4
FROM mcphersonsharyn_silver.manta_ray_silver
WHERE locality = 'Raja Ampat'
)
GROUP BY year, enso_phase
)
SELECT
year,
enso_phase,
avg_sst,
avg_cc,
-- Compare each year's SST to the earliest year in the dataset
avg_sst - FIRST_VALUE(avg_sst) OVER (
ORDER BY year
) AS sst_change_from_baseline,
-- Flag if this La Nina year is warmer than any El Nino year
CASE
WHEN enso_phase = 'La Nina'
AND avg_sst > (SELECT max(avg_sst) FROM enso_sst WHERE enso_phase = 'El Nino')
THEN 'La Nina WARMER than El Nino peak — significant warming signal'
WHEN enso_phase = 'La Nina'
AND avg_sst > (SELECT avg(avg_sst) FROM enso_sst WHERE enso_phase = 'El Nino')
THEN 'La Nina WARMER than El Nino average — warming trend present'
ELSE 'Within expected ENSO range'
END AS warming_signal
FROM enso_sst
ORDER BY year;


-- ─────────────────────────────────────────────────────────────
-- Query 4: Same analysis but for ALL species and localities
-- not just Raja Ampat — tests whether the pattern
-- is specific to Indonesia or global across your study region
-- ─────────────────────────────────────────────────────────────

WITH species_sst AS (
SELECT
m.species_common,
m.locality,
e.year,
e.enso_phase,
round(avg(e.sea_surface_temperature), 2) AS avg_sst,
round(avg(e.chlorophyll_concentration), 4) AS avg_cc
FROM mcphersonsharyn_silver.migration_silver m
LEFT JOIN mcphersonsharyn_silver.environmental_conditions_silver e
ON m.h3_res4 = e.h3_res4
AND m.year = e.year
AND m.month = e.month
WHERE e.sea_surface_temperature IS NOT NULL
GROUP BY m.species_common, m.locality, e.year, e.enso_phase
)
SELECT
species_common,
locality,
year,
enso_phase,
avg_sst,
avg_cc,
avg_sst - FIRST_VALUE(avg_sst) OVER (
PARTITION BY species_common, locality
ORDER BY year
) AS sst_change_from_first_year,
CASE
WHEN avg_sst - FIRST_VALUE(avg_sst) OVER (
PARTITION BY species_common, locality ORDER BY year
) > 1.0 THEN 'Warming above 1C from baseline'
WHEN avg_sst - FIRST_VALUE(avg_sst) OVER (
PARTITION BY species_common, locality ORDER BY year
) > 0.5 THEN 'Warming above 0.5C from baseline'
WHEN avg_sst - FIRST_VALUE(avg_sst) OVER (
PARTITION BY species_common, locality ORDER BY year
) < 0 THEN 'Cooler than baseline'
ELSE 'Within 0.5C of baseline'
END AS warming_category
FROM species_sst
ORDER BY species_common, locality, year;


-- ─────────────────────────────────────────────────────────────
-- Query 5: The definitive test
-- Average SST by ENSO phase across your entire study period
-- If La Nina average SST across all years is approaching
-- or exceeding El Nino average SST — that is your finding
-- ─────────────────────────────────────────────────────────────

SELECT
enso_phase,
count(DISTINCT year) AS years_in_phase,
round(avg(sea_surface_temperature), 2) AS overall_avg_sst,
round(min(sea_surface_temperature), 2) AS min_sst,
round(max(sea_surface_temperature), 2) AS max_sst,
round(avg(chlorophyll_concentration), 4) AS overall_avg_cc,
min(year) AS earliest_year,
max(year) AS latest_year
FROM mcphersonsharyn_silver.environmental_conditions_silver
WHERE h3_res4 IN (
SELECT DISTINCT h3_res4
FROM mcphersonsharyn_silver.manta_ray_silver
WHERE locality = 'Raja Ampat'
)
GROUP BY enso_phase
ORDER BY overall_avg_sst DESC;